# syft-widget Development Demo

This shows the new development-first approach to syft-widget.

## 1. Quick Start - Development Mode

Just import and start - no setup required!

In [ ]:
import sys
sys.path.append('..')

from syft_widget import start_infrastructure, stop_infrastructure
from syft_widget import APIDisplay, register_endpoint

In [ ]:
# Start in development mode - no SyftBox, no file generation
start_infrastructure(mode="development")

## 2. Create Custom Widgets

The new pattern: register endpoints, then create widgets that use them.

In [ ]:
# Define custom endpoints
@register_endpoint("/demo/counter")
def get_counter():
    import random
    return {"count": random.randint(1, 100)}

@register_endpoint("/demo/status")
def get_status():
    import time
    return {
        "status": "running",
        "timestamp": int(time.time()),
        "uptime": "5m 23s"
    }

In [ ]:
# Create a simple widget
class CounterWidget(APIDisplay):
    def __init__(self):
        super().__init__(endpoints=["/demo/counter"])
    
    def render_content(self, data, server_type="checkpoint"):
        counter = data.get("/demo/counter", {})
        count = counter.get("count", 0)
        
        # Server type badge
        badge_color = {"checkpoint": "#6c757d", "thread": "#28a745", "syftbox": "#007bff"}.get(server_type, "#6c757d")
        server_label = {"checkpoint": "📁 Checkpoint", "thread": "🧵 Thread Server", "syftbox": "📦 SyftBox"}.get(server_type, server_type)
        
        return f'''
        <div style="font-family: -apple-system, sans-serif; padding: 20px; background: #e3f2fd; border-radius: 8px; position: relative;">
            <div style="position: absolute; top: 10px; right: 10px; background: {badge_color}; color: white; padding: 4px 8px; border-radius: 4px; font-size: 12px;">
                {server_label}
            </div>
            
            <h3 style="margin: 0 0 15px 0;">🔢 Counter Widget</h3>
            <div style="font-size: 48px; font-weight: bold; color: #1976d2; text-align: center;">
                {count}
            </div>
            <div style="text-align: center; color: #666; margin-top: 10px;">
                Random counter value
            </div>
        </div>
        '''
    
    def get_update_script(self):
        return '''
        const counter = currentData["/demo/counter"] || {};
        const count = counter.count || 0;
        
        // Server type badge
        const badgeColors = {checkpoint: "#6c757d", thread: "#28a745", syftbox: "#007bff"};
        const serverLabels = {checkpoint: "📁 Checkpoint", thread: "🧵 Thread Server", syftbox: "📦 SyftBox"};
        const badgeColor = badgeColors[currentServerType] || "#6c757d";
        const serverLabel = serverLabels[currentServerType] || currentServerType;
        
        element.innerHTML = `
        <div style="font-family: -apple-system, sans-serif; padding: 20px; background: #e3f2fd; border-radius: 8px; position: relative;">
            <div style="position: absolute; top: 10px; right: 10px; background: ${badgeColor}; color: white; padding: 4px 8px; border-radius: 4px; font-size: 12px;">
                ${serverLabel}
            </div>
            
            <h3 style="margin: 0 0 15px 0;">🔢 Counter Widget</h3>
            <div style="font-size: 48px; font-weight: bold; color: #1976d2; text-align: center;">
                ${count}
            </div>
            <div style="text-align: center; color: #666; margin-top: 10px;">
                Random counter value
            </div>
        </div>
        `;
        '''

# Display it
counter = CounterWidget()
counter

In [ ]:
# Create another widget using multiple endpoints
class StatusWidget(APIDisplay):
    def __init__(self):
        super().__init__(endpoints=["/demo/status", "/demo/counter"])
    
    def render_content(self, data, server_type="checkpoint"):
        status = data.get("/demo/status", {})
        counter = data.get("/demo/counter", {})
        
        badge_color = {"checkpoint": "#6c757d", "thread": "#28a745", "syftbox": "#007bff"}.get(server_type, "#6c757d")
        server_label = {"checkpoint": "📁 Checkpoint", "thread": "🧵 Thread Server", "syftbox": "📦 SyftBox"}.get(server_type, server_type)
        
        return f'''
        <div style="font-family: -apple-system, sans-serif; padding: 20px; background: #f3e5f5; border-radius: 8px; position: relative;">
            <div style="position: absolute; top: 10px; right: 10px; background: {badge_color}; color: white; padding: 4px 8px; border-radius: 4px; font-size: 12px;">
                {server_label}
            </div>
            
            <h3 style="margin: 0 0 15px 0;">📊 Status Dashboard</h3>
            
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px;">
                <div>
                    <strong>Status:</strong> {status.get("status", "unknown").upper()}<br>
                    <strong>Uptime:</strong> {status.get("uptime", "...")}<br>
                    <strong>Timestamp:</strong> {status.get("timestamp", "...")}
                </div>
                <div style="text-align: center;">
                    <div style="font-size: 32px; font-weight: bold; color: #7b1fa2;">
                        {counter.get("count", "N/A")}
                    </div>
                    <div style="color: #666; font-size: 14px;">Current count</div>
                </div>
            </div>
        </div>
        '''

# Display it
dashboard = StatusWidget()
dashboard

## 3. Development vs Production

The key difference:

In [ ]:
print("Development mode benefits:")
print("✅ No file generation")
print("✅ No SyftBox required")
print("✅ Just checkpoint → thread server")
print("✅ Perfect for prototyping")
print()
print("Production mode adds:")
print("🏭 Generates run_widgets.sh")
print("🏭 Full checkpoint → thread → SyftBox lifecycle")
print("🏭 Automatic SyftBox app deployment")

## 4. Available Demo Widgets

The package still includes demo widgets, but now they're clearly examples:

In [ ]:
# These are still available but now optional
from syft_widget import TimeDisplay, CPUDisplay

# Quick demo
time_widget = TimeDisplay()
time_widget

## 5. Clean Up

In [ ]:
stop_infrastructure()

## Summary

The new syft-widget architecture:

1. **Development-first**: `start_infrastructure(mode="development")` just works
2. **True library**: Import and use, don't fork and modify
3. **Progressive complexity**: Add production features when ready
4. **Clean packages**: No file pollution in development mode
5. **Flexible deployment**: Works standalone or with SyftBox

This makes syft-widget much more developer-friendly while preserving all the SyftBox integration features for those who need them.